# Task 7: Gradio UI로 Posts 데이터 실시간 필터링

## 과제 목표
1. `https://jsonplaceholder.typicode.com/posts`에서 데이터를 불러온다.
2. `requests.get(url, params=params)`를 사용하여 **서버 사이드 필터링**을 수행한다.
3. 사용자가 **Gradio UI**를 통해 `userId`를 입력하면 실시간으로 해당 데이터를 필터링하여 DataFrame으로 보여준다.

## 학습 목표
1. `requests.get()`의 **`params` 파라미터**가 쿼리스트링으로 변환되는 원리를 이해한다.
2. **서버 사이드 필터링** vs **클라이언트 사이드 필터링**의 차이를 배운다.
3. **Gradio** 라이브러리로 간단한 웹 UI를 만드는 방법을 익힌다.
4. 함수 → UI 변환이라는 Gradio의 핵심 패러다임을 이해한다.

---
## 1단계: 환경 설정

### Gradio란?

Gradio는 **Python 함수를 웹 UI로 감싸주는** 라이브러리입니다.
머신러닝 모델의 데모를 만들기 위해 시작되었지만, 현재는 **데이터 앱, API 테스터, 교육용 도구** 등
다양한 용도로 사용됩니다.

### Gradio의 핵심 철학: "함수 → UI"

```
일반 Python 함수  →  Gradio  →  웹 인터페이스
f(x) → y          감싸기      입력 위젯 → 출력 위젯
```

예를 들어:
- 함수의 **입력 파라미터** → UI의 **입력 위젯** (텍스트 박스, 슬라이더 등)
- 함수의 **반환값** → UI의 **출력 위젯** (텍스트, 표, 이미지 등)

HTML/CSS/JavaScript를 전혀 몰라도 **파이썬만으로** 웹 앱을 만들 수 있습니다.

In [ ]:
import requests
import pandas as pd
import gradio as gr

print(f"requests: {requests.__version__}")
print(f"pandas: {pd.__version__}")
print(f"gradio: {gr.__version__}")
print("환경 설정 완료!")

---
## 2단계: `params`를 사용한 서버 사이드 필터링 이해

### `params` 딕셔너리 → 쿼리스트링 변환

`requests.get(url, params={...})`에서 `params` 딕셔너리는 **자동으로 URL 쿼리스트링**으로 변환됩니다:

```python
requests.get(
    "https://jsonplaceholder.typicode.com/posts",
    params={"userId": 2}
)
```

이 코드는 내부적으로 아래 URL로 요청합니다:
```
https://jsonplaceholder.typicode.com/posts?userId=2
                                          ↑ 쿼리스트링
```

### 직접 URL 조합 vs `params` 사용 비교

| 방법 | 코드 | 문제점 |
|------|------|--------|
| 직접 조합 | `f"{url}?userId={user_id}"` | 특수문자/한글 인코딩 누락 위험 |
| `params` 사용 | `requests.get(url, params={"userId": user_id})` | **자동 URL 인코딩**, 안전하고 깔끔 |

예를 들어 검색어에 한글이나 공백이 있다면:
- 직접 조합: `?q=서울 강남` → **에러 발생** (공백은 URL에서 허용되지 않음)
- `params`: `params={"q": "서울 강남"}` → 자동으로 `?q=%EC%84%9C%EC%9A%B8+%EA%B0%95%EB%82%A8`으로 인코딩

### 서버 사이드 필터링 vs 클라이언트 사이드 필터링

| 방식 | 동작 | 장점 | 단점 |
|------|------|------|------|
| **서버 사이드** | 서버가 필터링된 결과만 보내줌 | 네트워크 트래픽 절약, 대용량 데이터에 적합 | 매 요청마다 서버 부하 |
| **클라이언트 사이드** | 전체 데이터를 받은 후 로컬에서 필터 | 서버 부하 감소, 빠른 필터 전환 | 초기 로딩 느림, 메모리 사용 |

이 과제에서는 `params`를 사용한 **서버 사이드 필터링**을 수행합니다.

In [ ]:
# params를 사용한 서버 사이드 필터링 예시
url = "https://jsonplaceholder.typicode.com/posts"

# userId=2인 게시글만 요청
params = {"userId": 2}
response = requests.get(url, params=params)

print(f"요청 URL: {response.url}")  # 실제로 요청된 URL 확인
print(f"상태 코드: {response.status_code}")
print(f"반환된 게시글 수: {len(response.json())}개")

# 전체 데이터(100개) vs 필터링된 데이터(10개) 비교
all_response = requests.get(url)
print(f"\n전체 게시글: {len(all_response.json())}개")
print(f"userId=2 게시글: {len(response.json())}개")
print("→ 서버가 필터링된 결과만 보내줌!")

---
## 3단계: 데이터 수집 함수 작성

### 함수로 분리하는 이유

Gradio는 **Python 함수를 UI로 변환**합니다. 따라서:
1. 입력: `userId` (정수)
2. 처리: API 호출 + DataFrame 변환
3. 출력: DataFrame + 상태 메시지

이 세 단계를 하나의 함수로 묶어야 Gradio에 연결할 수 있습니다.

### 반환값이 여러 개인 이유

PDF 이미지(page 4)를 보면 UI에 두 가지 출력이 있습니다:
1. **네트워크 상태 및 결과 요약** (텍스트)
2. **수집된 데이터 (Pandas DataFrame)** (표)

Python에서 여러 값을 반환할 때는 **튜플(tuple)**로 반환합니다:
```python
return status_text, dataframe  # → (str, pd.DataFrame)
```

Gradio는 반환값의 순서와 출력 컴포넌트의 순서를 **자동으로 매핑**합니다.

In [ ]:
def fetch_posts_by_user(user_id):
    """
    userId로 JSONPlaceholder posts를 필터링하여 반환합니다.
    
    Parameters:
        user_id (int): 조회할 사용자 ID (1~10)
    
    Returns:
        tuple: (상태 메시지, DataFrame)
    """
    url = "https://jsonplaceholder.typicode.com/posts"
    
    # 입력값 검증
    user_id = int(user_id)
    if user_id < 1 or user_id > 10:
        return "⚠️ userId는 1~10 사이의 정수여야 합니다.", pd.DataFrame()
    
    # params를 사용한 서버 사이드 필터링
    params = {"userId": user_id}
    response = requests.get(url, params=params)
    
    # 상태 메시지 구성
    status = f"성공적으로 {len(response.json())}개의 행을 불러왔습니다. (상태 코드: {response.status_code})"
    
    # DataFrame 변환
    df = pd.DataFrame(response.json())
    
    return status, df

# 함수 테스트
status, df_test = fetch_posts_by_user(2)
print(status)
df_test.head(3)

---
## 4단계: Gradio UI 구성

### Gradio Blocks vs Interface

Gradio는 두 가지 API 스타일을 제공합니다:

| API | 코드량 | 유연성 | 적합한 상황 |
|-----|:------:|:------:|------------|
| `gr.Interface` | 적음 | 낮음 | 입력→출력이 단순한 경우 |
| `gr.Blocks` | 많음 | 높음 | 복잡한 레이아웃, 다중 컴포넌트 |

이 과제에서는 PDF 이미지와 동일한 UI를 구현하기 위해 **`gr.Blocks`**를 사용합니다.

### PDF 이미지 기반 UI 구조 분석

```
┌─────────────────────────────────────────────────┐
│  🌐 네트워크 & 웹 데이터 수집 실습               │  ← 제목
│  userId를 입력하여 REST API로부터...              │  ← 설명
├───────────────────────────┬─────────────────────┤
│  조회할 userId 입력 (1~10) │  [데이터 가져오기]   │  ← 입력 + 버튼
│  [    2                 ] │                     │
├───────────────────────────┴─────────────────────┤
│  네트워크 상태 및 결과 요약                       │  ← 상태 표시
│  성공적으로 10개의 행을 불러왔습니다. (200)       │
├─────────────────────────────────────────────────┤
│  수집된 데이터 (Pandas DataFrame)                 │  ← DataFrame 표
│  userId | id | title | body                     │
│  2      | 11 | et ea vero...  | delectus...     │
└─────────────────────────────────────────────────┘
```

### `gr.Blocks()` 컴포넌트 설명

| 컴포넌트 | 역할 | 대응 HTML |
|----------|------|-----------|
| `gr.Markdown()` | 마크다운 텍스트 표시 | `<h1>`, `<p>` 등 |
| `gr.Number()` | 숫자 입력 필드 | `<input type="number">` |
| `gr.Button()` | 클릭 버튼 | `<button>` |
| `gr.Textbox()` | 텍스트 출력 | `<textarea>` |
| `gr.Dataframe()` | pandas DataFrame 표시 | `<table>` |
| `gr.Row()` | 수평 배치 컨테이너 | `<div style="display:flex">` |

### `.click()` 이벤트 바인딩

```python
button.click(
    fn=함수,          # 실행할 Python 함수
    inputs=[입력들],   # 함수에 전달할 입력 컴포넌트
    outputs=[출력들]   # 함수 반환값을 표시할 출력 컴포넌트
)
```

버튼을 클릭하면 → `inputs`의 현재 값을 `fn`에 전달 → 반환값을 `outputs`에 표시

In [ ]:
# Gradio UI 구성
with gr.Blocks(title="네트워크 & 웹 데이터 수집 실습") as demo:
    
    # 제목 및 설명
    gr.Markdown(
        "## 🌐 네트워크 & 웹 데이터 수집 실습\n"
        "userId를 입력하여 REST API로부터 특정 사용자의 데이터를 Pandas 데이터프레임으로 가져옵니다."
    )
    
    # 입력 영역: userId 입력 + 버튼을 같은 행에 배치
    with gr.Row():
        user_input = gr.Number(
            label="조회할 userId 입력 (1~10)",
            value=1,              # 기본값
            minimum=1,            # 최솟값
            maximum=10,           # 최댓값
            precision=0,          # 정수만 허용 (소수점 0자리)
            scale=3               # 너비 비율 (버튼의 3배)
        )
        fetch_btn = gr.Button(
            "데이터 가져오기",
            variant="primary",    # 주요 버튼 스타일 (색상 강조)
            scale=1               # 너비 비율
        )
    
    # 상태 메시지 출력
    status_output = gr.Textbox(
        label="네트워크 상태 및 결과 요약",
        interactive=False         # 읽기 전용 (사용자 편집 불가)
    )
    
    # DataFrame 출력
    df_output = gr.Dataframe(
        label="수집된 데이터 (Pandas DataFrame)"
    )
    
    # 버튼 클릭 이벤트 바인딩
    fetch_btn.click(
        fn=fetch_posts_by_user,     # 실행할 함수
        inputs=[user_input],         # 함수에 전달할 입력
        outputs=[status_output, df_output]  # 반환값 표시 위치
    )

# UI 실행
demo.launch()

---
## 핵심 정리

### 이번 과제에서 배운 것

| 개념 | 핵심 내용 |
|------|----------|
| **`params` 파라미터** | 딕셔너리를 URL 쿼리스트링으로 자동 변환. 특수문자 인코딩 자동 처리 |
| **서버 사이드 필터링** | 서버가 조건에 맞는 데이터만 반환. 네트워크 트래픽 절약 |
| **Gradio Blocks** | Python 함수를 웹 UI로 변환. `Row()`로 레이아웃 구성 |
| **이벤트 바인딩** | `button.click(fn, inputs, outputs)`로 함수와 UI 연결 |

### 핵심 코드 흐름

```
사용자 입력 (userId)
    ↓
requests.get(url, params={"userId": n})
    ↓
서버가 필터링된 JSON 반환
    ↓
pd.DataFrame() 변환
    ↓
Gradio UI에 표시
```